# Homework 1
## LLM Zoomcamp
### C Guy Slater
### June 19-21, 2026

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader, chunk_documents
# from toyaikit import Agent

# 1. Setup Environment
load_dotenv()
openai_client = OpenAI()

In [2]:
# https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md#preparation
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
docs = []

for file in files:
    doc = file.parse()
    docs.append(doc)

In [4]:
print(f"{len(docs)} lesson pages in the dataset.")

72 lesson pages in the dataset.


## Q1. How many lesson pages

How many lesson pages are in the dataset?

* 24
* ✅72 
* 240
* 720

```python 
len(docs) == 72
True
```

In [5]:
import json
from pprint import pprint

# print(json.dumps(docs[0], indent=2))
pprint(docs[0])

{'content': '# Introduction\n'
            '\n'
            'Video: [Watch this '
            'lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n'
            '\n'
            "In this module, we'll build a working Retrieval-Augmented\n"
            'Generation (RAG) system from scratch, step by step.\n'
            '\n'
            'We write everything in plain Python. We build a small search '
            'index by\n'
            'hand and call the LLM ourselves. I want you to see every piece '
            'first.\n'
            'That way you know what a framework does for you before you reach '
            'for\n'
            'one.\n'
            '\n'
            'Places where you can find me:\n'
            '\n'
            '- [My substack](https://alexeyondata.substack.com/)\n'
            '- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n'
            '- [X](https://x.com/Al_Grigor)\n'
            '\n'
            '## LLMs\n'
     

In [6]:
# for Q2 
# https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md#q2-indexing-and-searching
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(docs)

In [7]:
query_hw2 = "How does the agentic loop keep calling the model until it stops?"

"""
Signature:
index.search(
    query,
    filter_dict=None,
    boost_dict=None,
    num_results=10,
    output_ids=False,
)
"""

search_results = index.search(
    query=query_hw2,
    num_results=5
)
first_five_results = [result['filename'] for result in search_results]

for rank, res in enumerate(first_five_results, start=1):
    print(f"#{rank}: {res}")
# print(f"first 5 results: \n{[]}")


#1: 01-agentic-rag/lessons/14-agentic-loop.md
#2: 01-agentic-rag/lessons/15-frameworks.md
#3: 01-agentic-rag/lessons/13-function-calling.md
#4: 01-agentic-rag/lessons/11-agents-intro.md
#5: 01-agentic-rag/lessons/16-other-frameworks.md


## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* ✅`01-agentic-rag/lessons/14-agentic-loop.md`✅ 
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

In [8]:
# Q3 - implementing HW_RAGBase
# for timing it
import time 

INSTRUCTIONS = """
Provide accurate answers to the question(s) based on the provided context. 

If answer cannot be identified from the provided context, respond with: 

'I am unable to answer to this question accurately given my current knowledge base.'
"""
PROMPT_TEMPLATE='''
QUESTION: {question}

CONTEXT: {context}
'''.strip()

class HW_RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model="gpt-5.4-mini"
    ):
        self.index=index
        self.llm_client=llm_client
        self.instructions=instructions
        self.prompt_template=prompt_template
        self.model=model

    def search(self, query, num_results=3):
        
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):

        lines=[]
        for res in search_results:
            lines.append(f"File: {res['filename']}")
            lines.append(f"Content: {res['content']}")
            lines.append('')

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def llm(self, prompt):
        input_messages=[
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]
        start_time = time.time()
        
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )
        latency = time.time() - start_time
        return response.output_text, response.usage.input_tokens, latency

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, tokens, latency = self.llm(prompt)
        return answer, tokens, latency

## Q3. RAG

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000 ✅
* 70000
* 700000

In [9]:
# Answering Q3:
# https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md#q3-rag

question = "How does the agentic loop keep calling the model until it stops?"

q3_assistant = HW_RAGBase(index=index, llm_client=openai_client)

ans, tokens, latency = q3_assistant.rag(question)

print(f'The provided answer to sample question (Q3):\n{ans} ')
print(f'Num tokens: {tokens}')
print(f'time in seconds: {latency:.2f}')

The provided answer to sample question (Q3):
The loop keeps calling the model by using a `while True` loop and checking for function calls in each response.

Specifically:

- It sends the current `messages` history to the model.
- It looks through `response.output`.
- If it finds any `function_call` items, it runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- If there are no function calls, it breaks out of the loop.

So the stopping condition is:

- **keep looping while the model returns function calls**
- **stop when a turn returns no function calls**

In the code, that’s this check:

```python
if has_function_calls == False:
    break
```

The model decides whether more tool use is needed; the loop just keeps going until the model finally returns a normal message with no tool calls. 
Num tokens: 5728
time in seconds: 17.99


In [10]:
# Q4 
from gitsource import chunk_documents

chunks = chunk_documents(docs, size=2000, step=1000)

In [11]:
type(chunks)

list

In [12]:
len(chunks) == 295

True

## Q4. Chunking

How many chunks do you get?

* 70
* 295 ✅
* 1100
* 4500

```python
len(chunks)==295
True
```

In [13]:
# Q5 
# https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md#q5-rag-with-chunking
index_chunked = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunked.fit(chunks)

In [14]:
q3_chunk_assistant = HW_RAGBase(index=index_chunked, llm_client=openai_client)

ans_chunked, tokens_chunked, latency_chunked = q3_chunk_assistant.rag(question)

print(f'The provided answer to sample question (Q5):\n{ans_chunked} ')
print(f'Num tokens: {tokens_chunked}')
print(f'time in seconds: {latency_chunked:.2f}')

The provided answer to sample question (Q5):
The loop keeps calling the model by checking whether the model returned any `function_call` items in the response.

- If there is at least one `function_call`, the code runs the tool, appends the result to `messages`, and continues the `while True` loop.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is simply:

```python
if has_function_calls == False:
    break
```

This means the agent keeps iterating until the model gives a final message with no more tool calls. 
Num tokens: 1493
time in seconds: 2.39


In [15]:
print(f'original tokens: {tokens}\nchunked tokens: {tokens_chunked}\ndifference: {tokens-tokens_chunked}')

original tokens: 5728
chunked tokens: 1493
difference: 4235


## Q5
Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

* about the same
* 3× fewer ✅
* 10× fewer
* 30× fewer

In [16]:
f'About {tokens/tokens_chunked:.2f}x fewer input tokens for the chunked version'

'About 3.84x fewer input tokens for the chunked version'

In [18]:
# Q6 
!uv add toyaikit

Resolved 129 packages in 28ms
Checked 125 packages in 746ms


In [19]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [20]:
def search(query: str) -> str:
    """
    Search the course markdown lesson chunks for information. 
    Use this to retrieve context about course topics.
    """
    print(f"   -> [Agent Executing Search: '{query}']")
    results = index_chunked.search(query, num_results=3)
    
    lines = []
    for doc in results:
        lines.append(f"File: {doc['filename']}")
        lines.append(f"Content: {doc['content']}\n")
    return "\n".join(lines).strip()

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search)

agent_instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
"""

In [23]:
#  the runner - while loop orchestrator
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [25]:
agent_question = "How does the agentic loop work, and how is it different from plain RAG?"

In [26]:
print(f"QUESTION: {agent_question}")
print("Starting Agentic Loop...\n" + "-" * 50)

QUESTION: How does the agentic loop work, and how is it different from plain RAG?
Starting Agentic Loop...
--------------------------------------------------


In [29]:
result = runner.loop(
    prompt=agent_question,
    callback=callback,
)

print("-" * 50)
print(f"Total loop cost: ${result.cost}")
print("\n*** Count the [Agent Executing Search] lines above to answer Q6 ***")

-> Response received
   -> [Agent Executing Search: 'agentic loop RAG difference loop retrieve act observe course']


   -> [Agent Executing Search: 'agentic loop in course lessons retrieval augmented generation plain RAG']


   -> [Agent Executing Search: 'RAG agentic loop search tool plan act observe']


-> Response received


--------------------------------------------------
Total loop cost: $CostInfo(input_cost=Decimal('0.00371925'), output_cost=Decimal('0.0021375'), total_cost=Decimal('0.00585675'))

*** Count the [Agent Executing Search] lines above to answer Q6 ***


## Q6. Turning it into an agent

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4 ✅ (3 times)
* 10
* 20

```bash
-> Response received
  #1 -> [Agent Executing Search: 'agentic loop RAG difference loop retrieve act observe course']
Function call: search({"query":"agentic loop RAG difference loop retr...)
  2 -> [Agent Executing Search: 'agentic loop in course lessons retrieval augmented generation plain RAG']
Function call: search({"query":"agentic loop in course lessons retrie...)
   -> [Agent Executing Search: 'RAG agentic loop search tool plan act observe']
Function call: search({"query":"RAG agentic loop search tool plan act...)
-> Response received
```